# LOLA Google Colab DJ Studio (Heavy DSP & GPU Stems -> Lightweight Game Export)
Run heavy audio analysis, beatmatching, 3-band EQ swaps, Demucs stem separation, and LUFS mastering on Colab's free GPU/CPU, then export lightweight JSON + mastered `.ogg`/`.mp3` files for `game/audio/`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg rubberband-cli
!pip -q install librosa soundfile scipy pyrubberband pyloudnorm demucs

In [ ]:
from pathlib import Path
import json, numpy as np, librosa, soundfile as sf, pyrubberband as pyrb, pyloudnorm as pyln
from scipy.signal import butter, sosfilt

INPUT_DIR = Path('/content/tracks')
OUT_DIR = Path('/content/lola_dj_export')
INPUT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
if Path('/content/lola_colab_tracks.zip').is_file():
    !unzip -qo /content/lola_colab_tracks.zip -d /content/tracks
print('Tracks ready in', INPUT_DIR, '->', len(list(INPUT_DIR.glob('*'))), 'files')

In [ ]:
def analyze_track(path: Path):
    y, sr = librosa.load(path, sr=44100, mono=True)
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beats, sr=sr)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr).mean(axis=1)
    keys = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    est_key = f'{keys[int(np.argmax(chroma))]} Minor'
    return {
        'id': path.stem.lower().replace(' ', '_'),
        'title': path.stem.replace('_', ' ').title(),
        'bpm': round(float(np.atleast_1d(tempo)[0]), 1),
        'key': est_key,
        'beat_drop': round(float(beat_times[min(16, len(beat_times)-1)]), 2) if len(beat_times) else 0.0,
        'bridge': round(float(beat_times[len(beat_times)//2]), 2) if len(beat_times) else 0.0,
    }

catalog = [analyze_track(p) for p in sorted(INPUT_DIR.glob('*')) if p.suffix.lower() in {'.mp3', '.wav', '.ogg'}]
(OUT_DIR / 'music_analysis.json').write_text(json.dumps(catalog, indent=2))
print(f'Analyzed {len(catalog)} tracks.')

In [ ]:
def eq_band_mix(track_a: Path, track_b: Path, out_wav: Path, fade_sec: float = 4.0, target_lufs: float = -14.0):
    ya, sr = sf.read(track_a, always_2d=True)
    yb, _ = sf.read(track_b, always_2d=True)
    info_a, info_b = analyze_track(track_a), analyze_track(track_b)
    if info_b['bpm'] > 0 and info_a['bpm'] > 0:
        yb = pyrb.time_stretch(yb, sr, info_a['bpm'] / info_b['bpm'])
    n = min(int(fade_sec * sr), len(ya), len(yb))
    sos_hp = butter(4, 240.0, btype='highpass', fs=sr, output='sos')
    t = np.linspace(0.0, np.pi / 2.0, n)[:, None]
    yb_head_hp = sosfilt(sos_hp, yb[:n], axis=0)
    cross = ya[-n:] * np.cos(t) + yb_head_hp * np.sin(t)
    mix = np.vstack([ya[:-n], cross, yb[n:]])
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(mix)
    mix = pyln.normalize.loudness(mix, loudness, target_lufs)
    sf.write(out_wav, np.clip(mix, -0.98, 0.98), sr)

print('Ready: call eq_band_mix(track_a, track_b, OUT_DIR / "mix.wav")')

In [ ]:
import subprocess

def export_gapless_ogg(src_audio: Path, out_ogg: Path, quality: int = 5):
    """Convert audio to 44.1 kHz VBR OGG for gapless Ren'Py looping."""
    out_ogg.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'ffmpeg', '-y', '-i', str(src_audio),
        '-ar', '44100', '-c:a', 'libvorbis', '-q:a', str(quality), str(out_ogg)
    ], check=True)

def export_demucs_stems(track_path: Path, stem_root: Path = OUT_DIR / 'stems'):
    """Run Demucs 4-stem separation and export equal-length 44.1 kHz .ogg layers."""
    subprocess.run(['demucs', '-n', 'htdemucs', '-o', '/tmp/demucs', str(track_path)], check=True)
    src_dir = Path('/tmp/demucs/htdemucs') / track_path.stem
    mapping = {'bass.wav': 'layer_base.ogg', 'drums.wav': 'layer_beat.ogg', 'other.wav': 'layer_melody.ogg', 'vocals.wav': 'layer_ambient.ogg'}
    for wav_name, ogg_name in mapping.items():
        if (src_dir / wav_name).is_file():
            export_gapless_ogg(src_dir / wav_name, stem_root / track_path.stem / ogg_name)

tracks = [p for p in sorted(INPUT_DIR.glob('*')) if p.suffix.lower() in {'.mp3', '.wav', '.ogg'}]
for p in tracks:
    export_gapless_ogg(p, OUT_DIR / 'ogg' / f'{p.stem}.ogg')
for p in tracks[:2]:
    export_demucs_stems(p)
print(f'Exported {len(tracks)} .ogg tracks and {min(2, len(tracks))} Demucs 4-stem packs.')

In [ ]:
!zip -qr /content/lola_dj_export.zip /content/lola_dj_export
print('Download /content/lola_dj_export.zip and extract into game/audio/')